In [ ]:
#!/usr/bin/env python3
"""
MRI多模态数据集3D重映射处理脚本
功能：
1. 创建prob_idx索引JSON文件
2. 将1D数据重映射为3D体积
3. 保存为新的MAT文件
"""

import os
import json
import numpy as np
import h5py
from pathlib import Path
import scipy.io as sio
from tqdm import tqdm
import argparse


def load_mat_h5(path):
    """
    正确加载MATLAB HDF5格式的MAT文件
    自动处理MATLAB与Python之间的存储差异
    """
    data = {}
    with h5py.File(path, "r") as f:
        for k in f.keys():
            if not k.startswith("#"):
                v = f[k][()]
                if v.ndim > 1:
                    v = v.T  # 关键：转置多维数组以适配Python行优先
                data[k] = v
    return data


def revert_reshape_f_order(array, region):
    """
    使用Fortran顺序将1D数组重构回3D体积
    适用于转置后的数据，保持与原始MATLAB逻辑一致
    
    Parameters:
    - array: 1D或2D numpy数组，shape为(n_voxels,)或(n_voxels, n_features)
    - region: 3D布尔数组，定义有效体素位置
    
    Returns:
    - big_img: 重构的3D或4D数组
    """
    if array.ndim == 1:
        array = array[:, np.newaxis]
   
    num_features = array.shape[1]
    big_img = np.zeros((*region.shape, num_features), dtype=array.dtype)
    big_img = big_img.reshape(-1, array.shape[1], order='F')
   
    bool_mask = region.flatten(order='F').astype(bool)
    big_img[bool_mask] = array
   
    big_img = big_img.reshape((*region.shape, num_features), order='F')
    
    if num_features == 1:
        big_img = big_img.squeeze(axis=-1)
   
    return big_img


def create_index_json(data_dir, output_json="dataset_index.json"):
    """
    创建数据集索引JSON文件
    将文件名映射到prob_idx (1-38)
    """
    data_dir = Path(data_dir)
    mat_files = sorted(list(data_dir.glob("*.mat")))
    
    if len(mat_files) == 0:
        raise ValueError(f"No .mat files found in {data_dir}")
    
    print(f"Found {len(mat_files)} .mat files")
    
    # 创建索引映射
    index_mapping = {}
    for idx, filepath in enumerate(mat_files, start=1):
        index_mapping[idx] = {
            "prob_idx": idx,
            "filename": filepath.name,
            "filepath": str(filepath),
            "subject_id": filepath.stem  # 不带扩展名的文件名
        }
    
    # 保存JSON文件
    output_path = data_dir / output_json
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(index_mapping, f, indent=2, ensure_ascii=False)
    
    print(f"Index JSON saved to: {output_path}")
    
    # 打印索引表
    print("\nDataset Index:")
    print("-" * 50)
    for idx in sorted(index_mapping.keys()):
        info = index_mapping[idx]
        print(f"prob_idx={info['prob_idx']:2d}: {info['filename']}")
    
    return index_mapping


def process_single_subject_to_3d(mat_path, prob_idx, output_dir):
    """
    处理单个被试数据，将1D数据重映射为3D体积
    
    Parameters:
    - mat_path: 输入MAT文件路径
    - prob_idx: 被试索引(1-38)
    - output_dir: 输出目录
    """
    print(f"\nProcessing subject {prob_idx}: {Path(mat_path).name}")
    
    # 加载数据（自动转置）
    data = load_mat_h5(mat_path)
    
    # 提取必要的数据
    region = data['region'].astype(bool)  # (384, 336, 256)
    multidim_data = data['multidim_data']  # (n_voxels, 351)
    seg_one_hot = data['seg_one_hot']  # (102, n_voxels)
    big_seg = data['big_seg']  # (384, 336, 256)
    
    print(f"  Region shape: {region.shape}")
    print(f"  Multidim data shape: {multidim_data.shape}")
    print(f"  Seg one-hot shape: {seg_one_hot.shape}")
    
    # 1. 将multidim_data重映射为4D体积 (384, 336, 256, 351)
    print("  Remapping multidim_data to 4D volume...")
    data_4d = np.zeros((*region.shape, multidim_data.shape[1]), dtype=np.float32)
    
    for feature_idx in tqdm(range(multidim_data.shape[1]), desc="  Features", leave=False):
        feature_1d = multidim_data[:, feature_idx]
        data_4d[:, :, :, feature_idx] = revert_reshape_f_order(feature_1d, region)
    
    # 2. 将seg_one_hot转换为3D整数标签
    print("  Converting one-hot to integer labels...")
    # 找出每个体素的类别（one-hot编码的最大值位置）
    labels_1d = np.argmax(seg_one_hot, axis=0).astype(np.uint8)  # (n_voxels,)
    labels_3d = revert_reshape_f_order(labels_1d, region)
    
    # 3. 创建prob_idx的3D体积（每个体素都是相同的prob_idx值）
    print("  Creating prob_idx volume...")
    prob_idx_3d = np.ones(region.shape, dtype=np.uint8) * prob_idx
    # 非脑区域设为0
    prob_idx_3d[~region] = 0
    
    # 4. 准备保存数据
    # 需要转置回MATLAB格式（Fortran顺序）以保持一致性
    save_data = {
        'data': data_4d.T,  # 转置回MATLAB格式
        'region': labels_3d.T,  # 转置回MATLAB格式
        'prob_idx': prob_idx_3d.T,  # 转置回MATLAB格式
        'big_seg': big_seg.T  # 转置回MATLAB格式（用于验证）
    }
    
    # 5. 保存为新的MAT文件
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    input_filename = Path(mat_path).stem
    output_filename = f"{input_filename}_3d.mat"
    output_path = output_dir / output_filename
    
    print(f"  Saving to: {output_path}")
    
    # 使用HDF5格式保存（-v7.3格式）
    with h5py.File(output_path, 'w') as f:
        for key, value in save_data.items():
            f.create_dataset(key, data=value, compression='gzip', compression_opts=4)
    
    # 验证保存的数据
    print("  Verifying saved data...")
    verify_data = load_mat_h5(output_path)
    print(f"    Saved data shape: {verify_data['data'].shape}")
    print(f"    Saved region shape: {verify_data['region'].shape}")
    print(f"    Saved prob_idx shape: {verify_data['prob_idx'].shape}")
    print(f"    Saved big_seg shape: {verify_data['big_seg'].shape}")
    
    # 验证big_seg是否一致（检查转置是否正确）
    if np.array_equal(verify_data['big_seg'], big_seg):
        print("    ✓ big_seg verification passed!")
    else:
        print("    ✗ WARNING: big_seg mismatch - check transpose operations!")
    
    return output_path


def process_all_subjects(data_dir, output_base_dir=None):
    """
    处理所有被试数据
    
    Parameters:
    - data_dir: 包含所有MAT文件的目录
    - output_base_dir: 输出基础目录，默认为data_dir的父目录
    """
    data_dir = Path(data_dir)
    
    # 设置输出目录
    if output_base_dir is None:
        output_base_dir = data_dir.parent
    output_3d_dir = Path(output_base_dir) / "3D"
    
    print(f"Data directory: {data_dir}")
    print(f"Output 3D directory: {output_3d_dir}")
    
    # 步骤1：创建索引JSON文件
    print("\n" + "="*60)
    print("Step 1: Creating index JSON file")
    print("="*60)
    index_mapping = create_index_json(data_dir)
    
    # 步骤2：处理每个被试，创建3D重映射文件
    print("\n" + "="*60)
    print("Step 2: Processing subjects to 3D volumes")
    print("="*60)
    
    successful = []
    failed = []
    
    for prob_idx in sorted(index_mapping.keys()):
        info = index_mapping[prob_idx]
        try:
            output_path = process_single_subject_to_3d(
                mat_path=info['filepath'],
                prob_idx=prob_idx,
                output_dir=output_3d_dir
            )
            successful.append((prob_idx, output_path))
        except Exception as e:
            print(f"  ✗ Failed to process subject {prob_idx}: {str(e)}")
            failed.append((prob_idx, str(e)))
    
    # 打印处理结果摘要
    print("\n" + "="*60)
    print("Processing Summary")
    print("="*60)
    print(f"Successfully processed: {len(successful)}/{len(index_mapping)}")
    
    if failed:
        print(f"\nFailed subjects ({len(failed)}):")
        for prob_idx, error in failed:
            print(f"  - Subject {prob_idx}: {error}")
    
    print(f"\nOutput files saved to: {output_3d_dir}")
    print(f"Index JSON saved to: {data_dir / 'dataset_index.json'}")
    
    return successful, failed


def verify_3d_file(mat_3d_path):
    """
    验证3D重映射文件的完整性
    """
    print(f"\nVerifying 3D file: {mat_3d_path}")
    
    data = load_mat_h5(mat_3d_path)
    
    print("Loaded data keys:", list(data.keys()))
    print(f"  data shape: {data['data'].shape}")  # Should be (384, 336, 256, 351)
    print(f"  region shape: {data['region'].shape}")  # Should be (384, 336, 256)
    print(f"  prob_idx shape: {data['prob_idx'].shape}")  # Should be (384, 336, 256)
    print(f"  big_seg shape: {data['big_seg'].shape}")  # Should be (384, 336, 256)
    
    # 检查prob_idx的唯一值
    unique_prob_idx = np.unique(data['prob_idx'])
    print(f"  Unique prob_idx values: {unique_prob_idx}")
    
    # 检查region的标签范围
    unique_labels = np.unique(data['region'])
    print(f"  Unique region labels: {len(unique_labels)} classes")
    print(f"  Label range: {unique_labels.min()} - {unique_labels.max()}")
    
    # 检查非零体素数量
    non_zero_voxels = np.sum(data['region'] > 0)
    total_voxels = np.prod(data['region'].shape)
    print(f"  Non-zero voxels: {non_zero_voxels:,} / {total_voxels:,} ({100*non_zero_voxels/total_voxels:.2f}%)")
    
    return True


def main():
    parser = argparse.ArgumentParser(description='Process MRI dataset to 3D volumes')
    parser.add_argument('data_dir', type=str, help='Directory containing .mat files')
    parser.add_argument('--output-dir', type=str, default=None,
                        help='Output base directory (default: parent of data_dir)')
    parser.add_argument('--verify-only', type=str, default=None,
                        help='Only verify a specific 3D file')
    
    args = parser.parse_args()
    
    if args.verify_only:
        verify_3d_file(args.verify_only)
    else:
        process_all_subjects(args.data_dir, args.output_dir)



if __name__ == "__main__":
    # 直接指定路径并运行
    data_directory = r"W:\radiologie\mr-physik-data\Mitarbeiter\Jiayi\Brain_voxel_38_prob_alex_mert\mri_mat_onehot"
    process_all_subjects(data_directory)
    